In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from sqlalchemy import create_engine


In [45]:
df = run_query("""
    SELECT * 
    FROM geo.voterfile_042025 
    WHERE lat IS NOT NULL AND lon IS NOT NULL
""")

In [12]:
# Centered on Miami-Dade (approx. downtown Miami)
map_center = [25.7617, -80.1918]  # Miami coordinates

In [22]:
legend_html = '''
 <div style="
     position: fixed; 
     bottom: 50px; left: 50px; width: 180px; height: 120px; 
     background-color: white;
     border:2px solid grey; z-index:9999; font-size:14px;
     padding: 10px;
     ">
     <b>Heatmap Density</b><br>
     <i style="background: #00f; width: 18px; height: 10px; display: inline-block;"></i> Low<br>
     <i style="background: #0f0; width: 18px; height: 10px; display: inline-block;"></i> Medium<br>
     <i style="background: #ff0; width: 18px; height: 10px; display: inline-block;"></i> High<br>
     <i style="background: #f00; width: 18px; height: 10px; display: inline-block;"></i> Very High
 </div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

In [36]:
# Create heatmap

m = folium.Map(location=[25.77, -80.25], zoom_start=13)
HeatMap(data=df[['lat', 'lon']].dropna(), radius=10).add_to(m)

# Add custom legend
m.get_root().html.add_child(folium.Element(legend_html))

m.save("miami_heatmap.html")

In [48]:
import geopandas as gpd
import folium
from folium import GeoJson
from folium.plugins import HeatMap
from db import get_engine  # your reusable engine
from shapely import wkt

# 1. Load geocoded points
import pandas as pd
df = pd.read_sql("SELECT lat, lon FROM geo.voterfile_042025 WHERE lat IS NOT NULL AND lon IS NOT NULL", get_engine())

# 2. Load the shapefile as GeoDataFrame from PostGIS
gdf = gpd.read_postgis(
    "SELECT * FROM geo.miami_city_district", 
    con=get_engine(), 
    geom_col="geometry"
)

# 3. Create base map
m = folium.Map(location=[25.77, -80.25], zoom_start=13)

# 4. Add heatmap
HeatMap(data=df[['lat', 'lon']], radius=10).add_to(m)

# 5. Add shapefile overlay
# Add district boundaries in red
folium.GeoJson(
    gdf,
    style_function=lambda feature: {
        'fillColor': 'red',
        'color': 'red',
        'weight': 2,
        'fillOpacity': 0.1
    }
).add_to(m)

# 6. (Optional) Add custom legend
m.get_root().html.add_child(folium.Element(legend_html))

# 7. Save map
m.save("miami_heatmap_with_districts.html")